# WESAD + SSL TimePatch Transformer

In [ ]:
!pip -q install numpy scipy scikit-learn matplotlib torch tqdm pandas seaborn optuna

In [ ]:
import os
import json
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from itertools import product
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

DEVICE: cuda
GPU Memory: 12.9 GB


## Configuración y Rutas

In [ ]:
WESAD_ROOT = "/home/leoisidro/CICLOS/IX/TESIS/WESAD-PROCESAMIENTO/WESAD/WESAD"
RESULTS_DIR = "/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_gridsearch"
os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET_FS = 4
WINDOW_SECONDS = 60
WINDOW_STRIDE_SECONDS = 30
USE_LABELS = {1, 2, 3}
STRESS_LABEL = 2

Guardando resultados en: /home/leoisidro/CICLOS/X/PFC_II/PFC2/results_gridsearch
Config optimización GPU: {'max_batch_size': 32, 'max_emb_dim': 256, 'cache_size': 8}


## Carga de Datos WESAD

In [3]:
def resample_1d(x, orig_fs, target_fs):
    x = np.asarray(x).squeeze()
    if orig_fs == target_fs:
        return x.astype(np.float32)
    duration = len(x) / float(orig_fs)
    target_len = max(2, int(round(duration * target_fs)))
    return signal.resample(x, target_len).astype(np.float32)

def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

def load_wesad_subject(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    wrist = data["signal"]["wrist"]
    labels = np.asarray(data["label"]).astype(int).squeeze()

    bvp = np.asarray(wrist["BVP"]).squeeze()
    eda = np.asarray(wrist["EDA"]).squeeze()
    temp = np.asarray(wrist["TEMP"]).squeeze()

    fs_bvp, fs_eda, fs_temp, fs_label = 64, 4, 4, 700

    bvp = resample_1d(bvp, fs_bvp, TARGET_FS)
    eda = resample_1d(eda, fs_eda, TARGET_FS)
    temp = resample_1d(temp, fs_temp, TARGET_FS)
    labels = resample_1d(labels, fs_label, TARGET_FS).round().astype(int)

    L = min(len(bvp), len(eda), len(temp), len(labels))
    X = np.stack([
        robust_norm(bvp[:L]),
        robust_norm(eda[:L]),
        robust_norm(temp[:L]),
    ], axis=1)

    labels = labels[:L]
    keep = np.isin(labels, list(USE_LABELS))
    X = X[keep]
    labels = labels[keep]
    y = np.where(labels == STRESS_LABEL, 1, 0).astype(np.int64)
    return X, y

def segment_windows(X, y, window_seconds=60, stride_seconds=30, fs=4):
    win = int(window_seconds * fs)
    stride = int(stride_seconds * fs)
    Xw, yw = [], []

    for start in range(0, len(X) - win + 1, stride):
        end = start + win
        xw = X[start:end]
        yw_raw = y[start:end]
        frac = yw_raw.mean()
        if frac in [0.0, 1.0] or frac <= 0.2 or frac >= 0.8:
            Xw.append(xw.astype(np.float32))
            yw.append(int(frac >= 0.5))

    if len(Xw) == 0:
        return np.zeros((0, win, X.shape[1]), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    return np.stack(Xw), np.asarray(yw, dtype=np.int64)

def load_all_wesad(root):
    X_all, y_all, groups_all = [], [], []
    subject_dirs = sorted([d for d in os.listdir(root) if d.startswith("S")])

    for subj in subject_dirs:
        pkl_path = os.path.join(root, subj, f"{subj}.pkl")
        if not os.path.exists(pkl_path):
            print("Skipping:", pkl_path)
            continue
        X, y = load_wesad_subject(pkl_path)
        Xw, yw = segment_windows(X, y, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS, TARGET_FS)
        if len(Xw) == 0:
            continue
        sid = int(subj[1:])
        X_all.append(Xw)
        y_all.append(yw)
        groups_all.append(np.full(len(yw), sid))
        print(subj, Xw.shape, "stress ratio=", float(yw.mean()))

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    return X_all, y_all, groups_all

print("Cargando WESAD...")
X, y, groups = load_all_wesad(WESAD_ROOT)
print(f"X: {X.shape}, y: {y.shape}, groups: {groups.shape}")
print(f"Subjects: {np.unique(groups)}")
print(f"Stress ratio: {float(y.mean()):.3f}")

Cargando WESAD...
S10 (73, 240, 3) stress ratio= 0.3013698630136986
S11 (71, 240, 3) stress ratio= 0.30985915492957744
S13 (72, 240, 3) stress ratio= 0.2916666666666667
S14 (71, 240, 3) stress ratio= 0.30985915492957744
S15 (72, 240, 3) stress ratio= 0.2916666666666667
S16 (71, 240, 3) stress ratio= 0.30985915492957744
S17 (73, 240, 3) stress ratio= 0.3013698630136986
S2 (67, 240, 3) stress ratio= 0.29850746268656714
S3 (68, 240, 3) stress ratio= 0.29411764705882354
S4 (70, 240, 3) stress ratio= 0.2857142857142857
S5 (70, 240, 3) stress ratio= 0.2714285714285714
S6 (70, 240, 3) stress ratio= 0.3
S7 (71, 240, 3) stress ratio= 0.28169014084507044
S8 (71, 240, 3) stress ratio= 0.29577464788732394
S9 (70, 240, 3) stress ratio= 0.3
X: (1060, 240, 3), y: (1060,), groups: (1060,)
Subjects: [ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17]
Stress ratio: 0.296


# Carga de Datos Stress-Predict Dataset

In [ ]:
# --- Loader desde processed_data (BVP + EDA + TEMP) para usar con TimePatchTransformer
import pandas as pd
import numpy as np

def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

def resample_1d(signal, target_len):
    signal = np.asarray(signal, dtype=np.float32)
    if len(signal) == target_len:
        return signal
    if len(signal) < 2:
        return np.full(target_len, float(signal[0]) if len(signal) else 0.0, dtype=np.float32)
    xp = np.linspace(0, 1, len(signal))
    x = np.linspace(0, 1, target_len)
    return np.interp(x, xp, signal).astype(np.float32)

def build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=240):
    # meta columns (incluye Participant para excluirla de features)
    meta_cols = {"Participant","Activity","PSS_BEFORE","PSS_AFTER","BASELINE_LABEL","TASK_LABEL","axis"}
    frames = {}

    for name, path in sensor_files.items():
        df = pd.read_csv(path, index_col=0)
        # Asegurar columna limpia `Participant` desde el índice (evita duplicados)
        participants_index = df.index.astype(str)
        df = df.reset_index(drop=True)
        df["Participant"] = participants_index

        # Comprobar existencia de label_col
        if label_col not in df.columns:
            raise ValueError(f"label_col '{label_col}' no está en {path}; columnas: {list(df.columns)[:10]}")

        # Columnas de señal: todas las columnas numéricas excepto las meta
        sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity")]
        frames[name] = df[["Participant","Activity"] + sig_cols + [label_col]].copy()

    # Añadir window index por participant+Activity para emparejar ventanas
    for k, df in frames.items():
        df["window_idx"] = df.groupby(["Participant","Activity"]).cumcount()
        frames[k] = df

    # Construir llave para merge: Participant + Activity + window_idx
    for k in frames:
        frames[k]["_key"] = frames[k]["Participant"].astype(str) + "||" + frames[k]["Activity"].astype(str) + "||" + frames[k]["window_idx"].astype(str)

    # Intersección de llaves comunes entre sensores
    common_keys = set.intersection(*[set(frames[k]["_key"].values) for k in frames])
    common_keys = sorted(common_keys)

    X_list = []
    y_list = []
    participants_list = []

    for key in common_keys:
        channel_data = []
        label_val = None
        part = None
        skip = False

        for name, df in frames.items():
            row = df[df["_key"] == key]
            if row.shape[0] != 1:
                skip = True
                break
            row = row.iloc[0]
            # Seleccionar columnas de señal (las guardadas en frames[name])
            sig = row[[c for c in df.columns if c not in {"Participant","Activity","window_idx","_key",label_col} and c not in meta_cols]].values.astype(np.float32)
            sig = robust_norm(sig)
            sig = resample_1d(sig, target_length)
            channel_data.append(sig)

            if label_val is None:
                # Etiquetar basado en Activity en lugar de PSS_AFTER
                activity = row['Activity']
                if activity == 'Baseline':
                    label_val = 0
                else:  # Stroop o Interview
                    label_val = 1
            if part is None:
                part = row["Participant"]

        if skip:
            continue

        # channel_data: lista de arrays (T,) por sensor -> apilar como (T, C)
        X_list.append(np.stack(channel_data, axis=1))
        y_list.append(label_val)
        participants_list.append(part)

    if len(X_list) == 0:
        return np.zeros((0, target_length, len(frames),), dtype=np.float32), np.zeros((0,), dtype=np.int64), np.zeros((0,), dtype=np.int64), []

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    groups = np.array([int(p.lstrip("S")) for p in participants_list], dtype=np.int64)

    return X, y, groups, participants_list

sensor_files = {
    "BVP": "processed_data/BVP_1_min_ven_30.csv",
    "EDA": "processed_data/EDA_1_min_ven_30.csv",
    "TEMP": "processed_data/TEMP_1_min_ven_30.csv",
}

X_proc, y_proc, groups_proc, participants = build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=240)
print("Loaded processed_data -> X.shape:", X_proc.shape, "y.shape:", y_proc.shape, "groups.shape:", groups_proc.shape)

X1, y1, groups_1 = X_proc, y_proc, groups_proc

Loaded processed_data -> X.shape: (1454, 240, 3) y.shape: (1454,) groups.shape: (1454,)


## Generación de Datos Sintéticos

In [4]:
def estimate_class_stats(X, y):
    stats = {}
    for cls in [0, 1]:
        Xc = X[y == cls]
        flat = Xc.reshape(-1, X.shape[-1])
        mu = flat.mean(axis=0)
        cov = np.cov(flat.T) + 1e-4 * np.eye(X.shape[-1])

        amps = []
        for sample in Xc[:min(len(Xc), 300)]:
            sample_amp = []
            for ch in range(sample.shape[1]):
                sample_amp.append(np.abs(np.fft.rfft(sample[:, ch])))
            amps.append(np.stack(sample_amp, axis=0))
        amp = np.stack(amps, axis=0).mean(axis=0)
        stats[cls] = {"mu": mu, "cov": cov, "amp": amp}
    return stats

def smooth_curve(length, scale=1.0, knots=8):
    xp = np.linspace(0, length - 1, knots)
    yp = np.random.normal(0, scale, knots)
    curve = np.interp(np.arange(length), xp, yp)
    wl = max(5, (length // 15) | 1)
    return signal.savgol_filter(curve, wl, 2).astype(np.float32)

def generate_synthetic_window(label, length, stats):
    base = np.random.multivariate_normal(stats[label]["mu"], stats[label]["cov"], size=length).astype(np.float32)
    out = np.zeros_like(base)

    for c in range(base.shape[1]):
        phase = np.random.uniform(0, 2*np.pi, size=stats[label]["amp"].shape[-1])
        spec = stats[label]["amp"][c] * np.exp(1j * phase)
        seq = np.fft.irfft(spec, n=length).real.astype(np.float32)
        seq = (seq - seq.mean()) / (seq.std() + 1e-6)
        out[:, c] = 0.5 * base[:, c] + 0.5 * seq

    t = np.linspace(0, 1, length).astype(np.float32)
    freq = 1.1 + 0.35 * label + np.random.uniform(-0.12, 0.12)
    out[:, 0] += 0.35 * np.sin(2 * np.pi * freq * np.arange(length) / TARGET_FS).astype(np.float32)

    n_peaks = np.random.randint(4, 9) if label == 1 else np.random.randint(1, 4)
    idx = np.arange(length)
    for _ in range(n_peaks):
        center = np.random.randint(0, length)
        width = np.random.uniform(2, 10)
        amp = np.random.uniform(0.3, 1.0) * (1.2 if label == 1 else 0.8)
        out[:, 1] += amp * np.exp(-0.5 * ((idx - center) / width) ** 2)

    out[:, 2] += ((-0.35 if label == 1 else 0.08) * t + smooth_curve(length, scale=0.05)).astype(np.float32)

    for c in range(out.shape[1]):
        out[:, c] += 0.2 * smooth_curve(length, scale=0.12 + 0.03 * c)

    for c in range(out.shape[1]):
        med = np.median(out[:, c])
        iqr = np.percentile(out[:, c], 75) - np.percentile(out[:, c], 25)
        iqr = iqr if iqr > 1e-6 else out[:, c].std() + 1e-6
        out[:, c] = (out[:, c] - med) / iqr

    return out.astype(np.float32)

def generate_synthetic_dataset(n_samples, class_balance=0.5):
    ys = (np.random.rand(n_samples) < class_balance).astype(np.int64)
    Xs = np.stack([generate_synthetic_window(int(lbl), X.shape[1], CLASS_STATS) for lbl in ys], axis=0)
    return Xs, ys

print("Estimando estadísticas de clases...")
CLASS_STATS = estimate_class_stats(X, y)
print("Hecho.")

Estimando estadísticas de clases...
Hecho.


## Datasets y Augmentaciones Mejoradas

In [5]:
class WindowDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


def advanced_augment(x):
    """Augmentaciones avanzadas para señales fisiológicas"""
    x = x.clone().float()
    
    # 1. Jitter: ruido gaussiano
    if random.random() < 0.8:
        x = x + 0.02 * torch.randn_like(x)
    
    # 2. Scaling: escala amplitud por canal
    if random.random() < 0.6:
        scale = torch.empty((1, x.shape[1])).uniform_(0.85, 1.15)
        x = x * scale
    
    # 3. Time warping
    if random.random() < 0.4:
        rate = random.uniform(0.95, 1.05)
        target_len = int(x.shape[0] * rate)
        indices = torch.linspace(0, x.shape[0]-1, target_len)
        x = torch.nn.functional.interpolate(
            x.T.unsqueeze(0), size=target_len, mode='linear', align_corners=False
        ).squeeze(0).T
    
    # Asegurar longitud correcta
    if x.shape[0] != 240:
        if x.shape[0] < 240:
            x = F.pad(x, (0, 0, 0, 240 - x.shape[0]))
        else:
            x = x[:240]
    
    # 4. Masking
    if random.random() < 0.3:
        c = random.randrange(x.shape[1])
        start = random.randrange(0, max(1, x.shape[0] - 20))
        width = random.randrange(5, 20)
        x[start:min(start+width, x.shape[0]), c] = 0
    
    return x


class SSLPairDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        return advanced_augment(x), advanced_augment(x)

print("Datasets y augmentaciones cargados.")

Datasets y augmentaciones cargados.


## Focal Loss

In [6]:
class FocalLoss(nn.Module):
    """Focal Loss para manejar desbalance de clases"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        return focal.mean()

print("Focal Loss definido.")

Focal Loss definido.


## TimePatch Transformer

In [7]:
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128):
        super().__init__()
        self.patch_len = patch_len
        self.proj = nn.Linear(in_ch * patch_len, emb_dim)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :].reshape(B, T2 // P, P * C)
        return self.proj(x)


class TimePatchTransformer(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, depth=4, heads=4, num_classes=2, dropout=0.1):
        super().__init__()
        self.patch = PatchEmbed1D(in_ch, patch_len, emb_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, 512, emb_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(emb_dim)
        self.ssl_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, 64)
        )
        self.cls_head = nn.Linear(emb_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward_features(self, x):
        x = self.patch(x)
        B, N, D = x.shape
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :N+1, :]
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]

    def forward_ssl(self, x):
        z = self.forward_features(x)
        z = self.ssl_head(z)
        return F.normalize(z, dim=-1)

    def forward(self, x):
        z = self.forward_features(x)
        return self.cls_head(z)

print("TimePatchTransformer definido.")

TimePatchTransformer definido.


## Funciones de Entrenamiento y Evaluación

In [8]:
def nt_xent(z1, z2, temp=0.1):
    """NT-Xent loss para contrastive learning"""
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.arange(B, device=z.device)
    targets = torch.cat([targets + B, targets], dim=0)
    return F.cross_entropy(sim, targets)


def pretrain_ssl(model, X_unlabeled, epochs=6, batch_size=32, lr=5e-4):
    """Preentrenamiento SSL"""
    ds = SSLPairDataset(X_unlabeled)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5)

    model.train()
    for ep in range(epochs):
        losses = []
        pbar = tqdm(dl, desc=f"SSL {ep+1}/{epochs}", leave=False)
        for x1, x2 in pbar:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1 = model.forward_ssl(x1)
            z2 = model.forward_ssl(x2)
            loss = nt_xent(z1, z2, temp=0.1)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
            pbar.set_postfix({'loss': f'{np.mean(losses):.4f}'})
        print(f"SSL epoch {ep+1}/{epochs}: {np.mean(losses):.4f}")
    return model


def evaluate_probs(y_true, probs):
    """Calcula métricas de evaluación"""
    pred = (probs >= 0.5).astype(int)
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1": float(f1_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
    }
    try:
        out["auroc"] = float(roc_auc_score(y_true, probs))
    except Exception:
        out["auroc"] = np.nan
    return out


def smooth_predictions(probs, window_size=5):
    """Suaviza predicciones con filtro de mediana"""
    return np.array([np.median(probs[max(0,i-window_size//2):min(len(probs),i+window_size//2+1)])
                    for i in range(len(probs))])


def finetune(model, X_train, y_train, X_val, y_val, epochs=12, batch_size=32, lr=1e-4, use_focal=False):
    """Fine-tuning supervisado"""
    train_dl = DataLoader(WindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True, num_workers=0)
    val_dl = DataLoader(WindowDataset(X_val, y_val), batch_size=batch_size, shuffle=False, num_workers=0)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5)
    criterion = FocalLoss() if use_focal else nn.CrossEntropyLoss()

    best_state = None
    best_f1 = -1

    for ep in range(epochs):
        model.train()
        train_losses = []
        pbar = tqdm(train_dl, desc=f"FT {ep+1}/{epochs}", leave=False)

        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f'{np.mean(train_losses):.4f}'})

        model.eval()
        probs_all, ys_all = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(DEVICE)
                probs = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
                probs_all.append(probs)
                ys_all.append(yb.numpy())

        probs_all = np.concatenate(probs_all)
        ys_all = np.concatenate(ys_all)
        metrics = evaluate_probs(ys_all, probs_all)
        print(f"FT epoch {ep+1}/{epochs}: loss={np.mean(train_losses):.4f}, val_f1={metrics['f1']:.4f}")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model


def predict_probs(model, X_test, batch_size=64):
    """Genera predicciones"""
    dl = DataLoader(WindowDataset(X_test), batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    probs = []
    with torch.no_grad():
        for xb in dl:
            xb = xb.to(DEVICE)
            p = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)

print("Funciones de entrenamiento definidas.")

Funciones de entrenamiento definidas.


## LOSO Evaluation

In [9]:
def stratified_subsample(X, y, frac=0.2):
    """Submuestreo estratificado"""
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    n0 = max(1, int(len(idx0) * frac))
    n1 = max(1, int(len(idx1) * frac))
    keep = np.concatenate([
        np.random.choice(idx0, n0, replace=False),
        np.random.choice(idx1, n1, replace=False),
    ])
    np.random.shuffle(keep)
    return X[keep], y[keep]


def run_loso(X, y, groups, use_ssl=False, use_synth=False, use_focal=False,
             ssl_epochs=6, ft_epochs=12, synth_ratio=0.5, batch_size=32, 
             patch_len=8, emb_dim=128, depth=4, heads=4, dropout=0.1):
    """LOSO evaluation"""
    logo = LeaveOneGroupOut()
    results = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
        held_out = int(np.unique(groups[te_idx])[0])
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test, y_test = X[te_idx], y[te_idx]

        n_val = max(8, int(0.15 * len(X_train)))
        perm = np.random.permutation(len(X_train))
        val_idx = perm[:n_val]
        tr2_idx = perm[n_val:]

        X_tr, y_tr = X_train[tr2_idx], y_train[tr2_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        if use_synth:
            Xs, ys = generate_synthetic_dataset(int(len(X_tr) * synth_ratio), class_balance=float(y_tr.mean()))
            X_tr = np.concatenate([X_tr, Xs], axis=0)
            y_tr = np.concatenate([y_tr, ys], axis=0)

        model = TimePatchTransformer(
            in_ch=3, patch_len=patch_len, emb_dim=emb_dim, 
            depth=depth, heads=heads, num_classes=2, dropout=dropout
        )

        if use_ssl:
            X_unlab = X_train.copy()
            if use_synth:
                Xu, _ = generate_synthetic_dataset(int(0.5 * len(X_train)), class_balance=float(y_train.mean()))
                X_unlab = np.concatenate([X_unlab, Xu], axis=0)
            model = pretrain_ssl(model, X_unlab, epochs=ssl_epochs, batch_size=batch_size)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs, batch_size=batch_size, use_focal=use_focal)
        probs = predict_probs(model, X_test, batch_size=batch_size)
        probs = smooth_predictions(probs, window_size=5)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = held_out
        results.append(metrics)
        print(f"Fold {fold} (Subject {held_out}): {metrics}")

    return results


def summarize(name, results):
    """Resumen de resultados"""
    print("\n" + "="*80)
    print(name)
    print("="*80)
    summary = {}
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        vals = np.array([r[key] for r in results], dtype=float)
        mean_val = np.nanmean(vals)
        std_val = np.nanstd(vals)
        summary[key] = {"mean": float(mean_val), "std": float(std_val)}
        print(f"{key}: {mean_val:.4f} ± {std_val:.4f}")
    return summary

print("Funciones LOSO definidas.")

Funciones LOSO definidas.


In [ ]:
def run_loso_fast(X, y, groups, n_subjects=5, use_ssl=False, use_synth=False, use_focal=False,
                  ssl_epochs=4, ft_epochs=6, synth_ratio=0.5, batch_size=32, 
                  patch_len=8, emb_dim=128, depth=4, heads=4, dropout=0.1):
    """LOSO evaluation RÁPIDO con solo N sujetos"""
    logo = LeaveOneGroupOut()
    results = []
    
    fold_count = 0
    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
        if fold_count >= n_subjects:
            break
        
        held_out = int(np.unique(groups[te_idx])[0])
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test, y_test = X[te_idx], y[te_idx]

        n_val = max(8, int(0.15 * len(X_train)))
        perm = np.random.permutation(len(X_train))
        val_idx = perm[:n_val]
        tr2_idx = perm[n_val:]

        X_tr, y_tr = X_train[tr2_idx], y_train[tr2_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        if use_synth:
            Xs, ys = generate_synthetic_dataset(int(len(X_tr) * synth_ratio), class_balance=float(y_tr.mean()))
            X_tr = np.concatenate([X_tr, Xs], axis=0)
            y_tr = np.concatenate([y_tr, ys], axis=0)

        model = TimePatchTransformer(
            in_ch=3, patch_len=patch_len, emb_dim=emb_dim, 
            depth=depth, heads=heads, num_classes=2, dropout=dropout
        )

        if use_ssl:
            X_unlab = X_train.copy()
            if use_synth:
                Xu, _ = generate_synthetic_dataset(int(0.5 * len(X_train)), class_balance=float(y_train.mean()))
                X_unlab = np.concatenate([X_unlab, Xu], axis=0)
            model = pretrain_ssl(model, X_unlab, epochs=ssl_epochs, batch_size=batch_size)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs, batch_size=batch_size, use_focal=use_focal)
        probs = predict_probs(model, X_test, batch_size=batch_size)
        probs = smooth_predictions(probs, window_size=5)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = held_out
        results.append(metrics)
        print(f"Fold {fold} (Subject {held_out}): F1={metrics['f1']:.4f}")
        
        fold_count += 1

    return results

Total de combinaciones rápidas: 96
Tiempo estimado: 20-40 minutos en RTX 3060
Sujetos usados para prueba: 5 de los disponibles



In [ ]:
import os
import json
from datetime import datetime

import numpy as np
import torch
import optuna


def objective_optuna(trial, X, y, groups, n_subjects=10):
    params = {
        'patch_len': trial.suggest_categorical('patch_len', [8, 16]),
        'emb_dim': trial.suggest_categorical('emb_dim', [128, 256]),
        'depth': trial.suggest_int('depth', 3, 5),
        'dropout': trial.suggest_float('dropout', 0.05, 0.25),
        'ft_epochs': trial.suggest_int('ft_epochs', 6, 10),
        'ssl_epochs': trial.suggest_int('ft_epochs', 4, 8),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32]),
        'use_ssl': trial.suggest_categorical('use_ssl', [True]),
        'use_synth': trial.suggest_categorical('use_synth', [False]),
        'use_focal': trial.suggest_categorical('use_focal', [False]),
    }

    torch.cuda.empty_cache()
    results = run_loso_fast(
        X, y, groups,
        n_subjects=n_subjects,
        use_ssl=params['use_ssl'],
        use_synth=params['use_synth'],
        use_focal=params['use_focal'],
        ssl_epochs=params['ssl_epochs'],
        ft_epochs=params['ft_epochs'],
        synth_ratio=0.5,
        batch_size=params['batch_size'],
        patch_len=params['patch_len'],
        emb_dim=params['emb_dim'],
        depth=params['depth'],
        heads=4,
        dropout=params['dropout'],
    )

    mean_f1 = float(np.nanmean([r['f1'] for r in results]))
    trial.set_user_attr('mean_accuracy', float(np.nanmean([r['accuracy'] for r in results])))
    trial.set_user_attr('mean_auroc', float(np.nanmean([r.get('auroc', np.nan) for r in results])))
    trial.set_user_attr('params', params)
    return mean_f1


def run_optuna_search(X, y, groups, n_trials=20, n_subjects=10, study_name='wesad_optuna'):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler, study_name=study_name)

    def _objective(trial):
        return objective_optuna(trial, X, y, groups, n_subjects=n_subjects)

    study.optimize(_objective, n_trials=n_trials)

    print('\n' + '=' * 80)
    print('OPTUNA FINALIZADO')
    print('=' * 80)
    print(f'Best F1: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    best_path = os.path.join(RESULTS_DIR, f'optuna_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
    payload = {
        'best_value': float(study.best_value),
        'best_params': study.best_params,
        'trials': [
            {
                'number': t.number,
                'value': None if t.value is None else float(t.value),
                'params': t.params,
                'user_attrs': t.user_attrs,
                'state': str(t.state),
            }
            for t in study.trials
        ],
    }
    with open(best_path, 'w') as f:
        json.dump(payload, f, indent=2)

    print(f'Resultados guardados en: {best_path}')
    return study


print('Optuna listo. Ejecuta run_optuna_search(X, y, groups, n_trials=20, n_subjects=10) para comenzar.')

Optuna listo. Ejecuta run_optuna_search(X, y, groups, n_trials=20, n_subjects=10) para comenzar.


In [13]:
# EJECUTAR OPTUNA
# Ajusta n_trials y n_subjects según el tiempo disponible.
print("INICIANDO OPTUNA...")
study_optuna = run_optuna_search(
    X, y, groups,
    n_trials=10,
    n_subjects=10,
    study_name="wesad_optuna"
)

print("\nMejor F1:", study_optuna.best_value)
print("Mejores parámetros:", study_optuna.best_params)

[I 2026-05-13 16:27:20,826] A new study created in memory with name: wesad_optuna


INICIANDO OPTUNA...


SSL 1/4:   0%|          | 0/62 [00:00<?, ?it/s]

SSL epoch 1/4: 1.0014


SSL 2/4:   0%|          | 0/62 [00:00<?, ?it/s]

...

FT 8/9:   0%|          | 0/53 [00:00<?, ?it/s]

FT epoch 8/9: loss=0.0863, val_f1=0.9500


FT 9/9:   0%|          | 0/53 [00:00<?, ?it/s]

[I 2026-05-13 16:44:29,207] Trial 9 finished with value: 0.8467299226562137 and parameters: {'patch_len': 8, 'emb_dim': 256, 'depth': 3, 'dropout': 0.19592123566761283, 'ft_epochs': 9, 'batch_size': 16, 'use_ssl': True, 'use_synth': False, 'use_focal': False}. Best is trial 2 with value: 0.8941444851234982.


FT epoch 9/9: loss=0.0494, val_f1=0.9302
Fold 10 (Subject 11): F1=0.3704

OPTUNA FINALIZADO
Best F1: 0.8941
Best params: {'patch_len': 8, 'emb_dim': 128, 'depth': 3, 'dropout': 0.12327236865873835, 'ft_epochs': 8, 'batch_size': 16, 'use_ssl': True, 'use_synth': False, 'use_focal': False}
Resultados guardados en: /home/leoisidro/CICLOS/X/PFC_II/PFC2/results_gridsearch/optuna_20260513_164429.json

Mejor F1: 0.8941444851234982
Mejores parámetros: {'patch_len': 8, 'emb_dim': 128, 'depth': 3, 'dropout': 0.12327236865873835, 'ft_epochs': 8, 'batch_size': 16, 'use_ssl': True, 'use_synth': False, 'use_focal': False}


In [14]:
print("\nMejor F1:", study_optuna.best_value)
print("Mejores parámetros:", study_optuna.best_params)


Mejor F1: 0.8941444851234982
Mejores parámetros: {'patch_len': 8, 'emb_dim': 128, 'depth': 3, 'dropout': 0.12327236865873835, 'ft_epochs': 8, 'batch_size': 16, 'use_ssl': True, 'use_synth': False, 'use_focal': False}


In [ ]:
best_cfg = study_optuna.best_params

# Optuna solo devuelve los hiperparámetros que realmente se buscaron.
# Para evitar pasar None al modelo, completamos con valores por defecto.
final_cfg = {
    "use_ssl": best_cfg.get("use_ssl", True),
    "use_synth": best_cfg.get("use_synth", False),
    "use_focal": best_cfg.get("use_focal", False),
    "ssl_epochs": best_cfg.get("ssl_epochs", 4),
    "ft_epochs": best_cfg.get("ft_epochs", 8),
    "synth_ratio": best_cfg.get("synth_ratio", 0.5),
    "batch_size": best_cfg.get("batch_size", 32),
    "patch_len": best_cfg.get("patch_len", 8),
    "emb_dim": best_cfg.get("emb_dim", 128),
    "depth": best_cfg.get("depth", 4),
    "heads": best_cfg.get("heads", 4),
    "dropout": best_cfg.get("dropout", 0.1),
}

print("ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN")
print("=" * 80)
print(f"Configuración: {final_cfg}")

final_results = run_loso_fast(
    X, y, groups,
    n_subjects=15,
    use_ssl=final_cfg["use_ssl"],
    use_synth=final_cfg["use_synth"],
    use_focal=final_cfg["use_focal"],
    ssl_epochs=final_cfg["ssl_epochs"],
    ft_epochs=final_cfg["ft_epochs"],
    synth_ratio=final_cfg["synth_ratio"],
    batch_size=final_cfg["batch_size"],
    patch_len=final_cfg["patch_len"],
    emb_dim=final_cfg["emb_dim"],
    depth=final_cfg["depth"],
    heads=final_cfg["heads"],
    dropout=final_cfg["dropout"],
)

final_summary = summarize("SSL + supervised", final_results)
print("\nResumen final:")
for metric_name, metric_values in final_summary.items():
    print(f"{metric_name}: {metric_values['mean']:.4f} ± {metric_values['std']:.4f}")

ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN
Configuración: {'use_ssl': True, 'use_synth': False, 'use_focal': False, 'ssl_epochs': 4, 'ft_epochs': 8, 'synth_ratio': 0.5, 'batch_size': 16, 'patch_len': 8, 'emb_dim': 128, 'depth': 3, 'heads': 4, 'dropout': 0.12327236865873835}


SSL 1/4:   0%|          | 0/62 [00:00<?, ?it/s]

SSL epoch 1/4: 0.9245


SSL 2/4:   0%|          | 0/62 [00:00<?, ?it/s]

SSL epoch 2/4: 0.5688


...

FT epoch 6/8: loss=0.0715, val_f1=0.9231


FT 7/8:   0%|          | 0/53 [00:00<?, ?it/s]

FT epoch 7/8: loss=0.0468, val_f1=0.9474


FT 8/8:   0%|          | 0/53 [00:00<?, ?it/s]

FT epoch 8/8: loss=0.0275, val_f1=0.9375
Fold 15 (Subject 17): F1=0.0000

SSL + supervised
accuracy: 0.8858 ± 0.1510
f1: 0.7711 ± 0.3394
balanced_accuracy: 0.8611 ± 0.2036
auroc: 0.9054 ± 0.2377

Resumen final:
accuracy: 0.8858 ± 0.1510
f1: 0.7711 ± 0.3394
balanced_accuracy: 0.8611 ± 0.2036
auroc: 0.9054 ± 0.2377


In [ ]:
best_cfg = study_optuna.best_params

# Optuna solo devuelve los hiperparámetros que realmente se buscaron.
# Para evitar pasar None al modelo, completamos con valores por defecto.
final_cfg = {
    "use_ssl": best_cfg.get("use_ssl", True),
    "use_synth": best_cfg.get("use_synth", False),
    "use_focal": best_cfg.get("use_focal", False),
    "ssl_epochs": best_cfg.get("ssl_epochs", 4),
    "ft_epochs": best_cfg.get("ft_epochs", 8),
    "synth_ratio": best_cfg.get("synth_ratio", 0.5),
    "batch_size": best_cfg.get("batch_size", 32),
    "patch_len": best_cfg.get("patch_len", 8),
    "emb_dim": best_cfg.get("emb_dim", 128),
    "depth": best_cfg.get("depth", 4),
    "heads": best_cfg.get("heads", 4),
    "dropout": best_cfg.get("dropout", 0.1),
}

print("ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN")
print("=" * 80)
print(f"Configuración: {final_cfg}")

final_results = run_loso_fast(
    X1, y1, groups_1,
    n_subjects=33,
    use_ssl=final_cfg["use_ssl"],
    use_synth=final_cfg["use_synth"],
    use_focal=final_cfg["use_focal"],
    ssl_epochs=final_cfg["ssl_epochs"],
    ft_epochs=final_cfg["ft_epochs"],
    synth_ratio=final_cfg["synth_ratio"],
    batch_size=final_cfg["batch_size"],
    patch_len=final_cfg["patch_len"],
    emb_dim=final_cfg["emb_dim"],
    depth=final_cfg["depth"],
    heads=final_cfg["heads"],
    dropout=final_cfg["dropout"],
)

final_summary_2 = summarize("SSL + supervised", final_results)
print("\nResumen final:")
for metric_name, metric_values in final_summary_2.items():
    print(f"{metric_name}: {metric_values['mean']:.4f} ± {metric_values['std']:.4f}")

ENTRENAMIENTO FINAL CON LA MEJOR CONFIGURACIÓN
Configuración: {'use_ssl': True, 'use_synth': False, 'use_focal': False, 'ssl_epochs': 4, 'ft_epochs': 8, 'synth_ratio': 0.5, 'batch_size': 16, 'patch_len': 8, 'emb_dim': 128, 'depth': 3, 'heads': 4, 'dropout': 0.12327236865873835}


SSL 1/4:   0%|          | 0/88 [00:00<?, ?it/s]

SSL epoch 1/4: 0.8064


SSL 2/4:   0%|          | 0/88 [00:00<?, ?it/s]

SSL epoch 2/4: 0.3580


...

FT epoch 6/8: loss=0.5982, val_f1=0.7397


FT 7/8:   0%|          | 0/75 [00:00<?, ?it/s]

FT epoch 7/8: loss=0.5956, val_f1=0.7365


FT 8/8:   0%|          | 0/75 [00:00<?, ?it/s]

FT epoch 8/8: loss=0.5724, val_f1=0.7415
Fold 33 (Subject 35): F1=0.7778

SSL + supervised
accuracy: 0.6147 ± 0.1176
f1: 0.7313 ± 0.0834
balanced_accuracy: 0.5738 ± 0.1262
auroc: 0.6375 ± 0.2330

Resumen final:
accuracy: 0.6147 ± 0.1176
f1: 0.7313 ± 0.0834
balanced_accuracy: 0.5738 ± 0.1262
auroc: 0.6375 ± 0.2330


In [28]:
final_summary = summarize("SSL + supervised", final_results)
print("\nResumen final:")
for metric_name, metric_values in final_summary.items():
    print(f"{metric_name}: {metric_values['mean']:.4f} ± {metric_values['std']:.4f}")


SSL + supervised
accuracy: 0.6147 ± 0.1176
f1: 0.7313 ± 0.0834
balanced_accuracy: 0.5738 ± 0.1262
auroc: 0.6375 ± 0.2330

Resumen final:
accuracy: 0.6147 ± 0.1176
f1: 0.7313 ± 0.0834
balanced_accuracy: 0.5738 ± 0.1262
auroc: 0.6375 ± 0.2330
